In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import pickle
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '2_Propensities'))
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '4_Baselines' / '4.2_OutcomeModel'))
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '4_Baselines'))
import SASRec_class as sasrec
import MF_class as MF
import OM_class as OM

# 1 Loading Dataset and Propensities Model

In [21]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Sequels'

train = pd.read_csv(data_path / 'train.csv')
test = pd.read_csv(data_path / 'test.csv')
with open(data_path / 'id2info.pkl', 'rb') as f:
    id2info = pickle.load(f)

n_users = len(train['user_id'].unique())
n_items = len(train['item_id'].unique())

full_data = pd.concat([train, test], ignore_index=True)

In [22]:
MF_model = MF.MatrixFactorizationTorch(n_users, n_items, n_factors=25)
MF_model.load(path=base_artifacts / 'Propensity_Models' / 'MF_sequels.pt')
MF_model.eval()

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            7801
Number of items:            6384
Number of factors:          25
Learning rate:              0.0005
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           40
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 14:17:00


MatrixFactorizationTorch()

# 2 Build Ground Truth

In [23]:
results = {'title_A': [], 'title_B': [], 'causal_link': []}
chosen_ids = []
for item1 in id2info:
    for item2 in id2info:
        if item1 == item2:
            continue
        if id2info[item1]['series'] != id2info[item2]['series']:
            continue
        num1 = id2info[item1]['number']
        num2 = id2info[item2]['number']
        if num1 == num2:
            continue
        if num1 < num2:
            link = 1
        else:
            link = 0
        results['title_A'].append(id2info[item1]['title'])
        results['title_B'].append(id2info[item2]['title'])
        results['causal_link'].append(link)
        chosen_ids.append((item1, item2))

oracle = pd.DataFrame(results)

# 3 Defining Baselines

In [24]:
pivot_real = full_data.pivot(index='user_id', columns='item_id', values='interaction').fillna(0)
itemid_to_colidx_pivot_real = {item_id: col_idx for col_idx, item_id in enumerate(pivot_real.columns)}
pivot_real_np = pivot_real.values

Q_normalized = (MF_model.Q / torch.norm(MF_model.Q, dim=1, keepdim=True)).cpu().detach().numpy()

def cosimilarity(idx1, idx2):
    """Calculate cosine similarity between two items."""
    return np.dot(Q_normalized[idx1], Q_normalized[idx2])

def correlation(idx1, idx2):
    """Calculate correlation between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    if T.std() == 0 or Y.std() == 0:
        return 0
    return np.corrcoef(T, Y)[0, 1]

def diff_of_conditionals(idx1, idx2):
    """Calculate difference of conditionals P(Y|T) - P(Y|~T) between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    p_T = np.clip(np.mean(T), 1e-6, 1-1e-6)
    p_Y = np.mean(Y)
    p_TY = np.mean(T * Y)
    return p_TY / p_T - (p_Y - p_TY) / (1 - p_T)

def jacard_index(idx1, idx2):
    """Calculate Jaccard index between two items."""
    colidx1 = itemid_to_colidx_pivot_real[idx1]
    colidx2 = itemid_to_colidx_pivot_real[idx2]
    T = pivot_real_np[:, colidx1]
    Y = pivot_real_np[:, colidx2]
    intersection = np.sum((T > 0) & (Y > 0))
    union = np.sum((T > 0) | (Y > 0))
    if union == 0:
        return 0
    return intersection / union

# 4 SASRec

In [25]:
# Load SASRec model
model_path = base_artifacts / 'SASRec_Models'
with open(model_path / 'sasrec_goodreads_init_dict.pkl', 'rb') as f:
    init_dict_loaded = pickle.load(f)
sasrec_model = sasrec.SASRecTorch(**init_dict_loaded)
sasrec_model.load(model_path / 'sasrec_goodreads.pt')

Model loaded from /home/gouni/CausalI2I_artifacts/SASRec_Models/sasrec_goodreads.pt.
num_items:     6384
max_seq_len:   50
device:        cuda
batch_size:    2048
lr:            0.001
weight_decay:  0.0
num_epochs:    20
saved_at:      2026-01-06 09:58:04
note:          None


/home/gouni/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [26]:
def make_sequence(cause_id):
    PAD = sasrec_model.num_items
    L = sasrec_model.max_seq_len
    seq = torch.full((1, L), PAD, dtype=torch.long, device=sasrec_model.device)
    seq[0, -1] = cause_id
    return seq

In [27]:
sasrec_model.eval()

sasrec_scores = {}
candidates = torch.arange(0, sasrec_model.num_items, device=sasrec_model.device)
for pair in tqdm(chosen_ids):
    cause_id, effect_id = pair
    seq = make_sequence(cause_id)
    candidates_scores = sasrec_model.predict_scores(seq, candidates).detach().cpu().numpy()[0]
    sasrec_scores[pair] = candidates_scores[effect_id]

  0%|          | 0/7340 [00:00<?, ?it/s]

# 5 Outcome Model

In [28]:
user_embeddings = MF_model.P.detach().numpy()[:-1]
item_embeddings = MF_model.Q.detach().numpy()[:-1]
user_bias = MF_model.b_u.detach().numpy()[:-1]
item_bias = MF_model.b_i.detach().numpy()[:-1]

OM_model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
)

OM_model.load(path = base_artifacts / 'Outcome_Models' / f'OM_sequels.pt')

Loaded OutcomeModel summary:
Model:             OutcomeModel
Embedding dim:     25
Loss:              BCE
Learning rate:     0.0001
Weight decay:      0.0001
Batch size:        8192
Epochs:            40
Use AMP:           True
Timestamp:         2026-04-13 14:45:19
Note:              none


In [29]:
from joblib import Parallel, delayed

OM_model.eval()
test_users = np.asarray(test['user_id'].unique())

def score_pair(pair):
    cause_id, effect_id = pair
    with torch.inference_mode():
        mu1 = OM_model.predict_outcome(
            u_list=test_users, i=cause_id, j=effect_id, x=1
        ).cpu().numpy()
        mu0 = OM_model.predict_outcome(
            u_list=test_users, i=cause_id, j=effect_id, x=0
        ).cpu().numpy()
    return pair, {0: mu0, 1: mu1}

results = Parallel(n_jobs=-1, prefer="processes")(
    delayed(score_pair)(pair) for pair in tqdm(chosen_ids)
)

om_scores = dict(results)

  0%|          | 0/7340 [00:00<?, ?it/s]

# 6 Defining ATE

In [30]:
test_probs = MF_model.predict_prob(
        torch.tensor(test['user_id'].values, dtype=torch.long),
        torch.tensor(test['item_id'].values, dtype=torch.long)
    )
test_copy = test.copy()
if test_copy['timestamp'].dtype == 'O':
    test_copy['timestamp'] = pd.to_datetime(test_copy['timestamp'], errors='coerce').astype(np.int64) // 10**9
    test_copy['timestamp'] = test_copy['timestamp'].apply(lambda x: x if x > 0 else np.inf)

pivot_test_timestamp = test_copy.pivot(index='user_id', columns='item_id', values='timestamp').fillna(np.inf)
pivot_test_timestamp_np = pivot_test_timestamp.values
itemid_to_colidx = {id: i for i, id in enumerate(pivot_test_timestamp.columns)}

test_copy['probability'] = test_probs.cpu().detach().numpy()
pivot_test_pred = test_copy.pivot(index='user_id', columns='item_id', values='probability')
pivot_test_pred_np = pivot_test_pred.values

In [31]:
test_interaction_time_cols  = {
    item: pivot_test_timestamp_np[:, colidx]
    for item, colidx in itemid_to_colidx.items()
}

pred_cols = {
    item: pivot_test_pred_np[:, colidx]
    for item, colidx in itemid_to_colidx.items()
}

all_interaction_cols = {
    item: pivot_real_np[:, colidx]
    for item, colidx in itemid_to_colidx_pivot_real.items()
}

test_users = test['user_id'].unique()

In [32]:
def get_ATE(
    cause_item,
    effect_item,
    clip=0,
    drop_inverted=True,
    stabilized=True,
    mode="ipw",   # "ipw", "dr", "om"
):
    """
    Estimate ATE using IPW, DR, or OM-only.

    Parameters
    ----------
    mode : {"ipw", "dr", "om"}
        ipw : inverse propensity weighting only
        dr  : doubly robust estimator
        om  : outcome model only (plug-in)
    """

    # --------------------------------------------------
    # Filter inverted interactions
    # --------------------------------------------------

    users        = np.array(test_users)
    cause_times  = test_interaction_time_cols[cause_item]
    effect_times = test_interaction_time_cols[effect_item]
    pi           = pred_cols[cause_item]

    if drop_inverted:
        keep = np.where((cause_times <= effect_times) | (cause_times == np.inf))[0]

        users        = users[keep]
        cause_times  = cause_times[keep]
        effect_times = effect_times[keep]
        pi           = pi[keep]

    T = (cause_times < np.inf).astype(float)
    Y = (effect_times < np.inf).astype(float)

    n = len(T)

    pi = np.clip(pi, clip, 1 - clip)

    # --------------------------------------------------
    # Outcome model (if needed)
    # --------------------------------------------------

    if mode in {"dr", "om"}:
        mu1 = om_scores[(cause_item, effect_item)][1]
        mu0 = om_scores[(cause_item, effect_item)][0]
        if drop_inverted:
            mu1 = mu1[keep]
            mu0 = mu0[keep]
        base = mu1 - mu0
    else:
        base = np.zeros(n)

    # --------------------------------------------------
    # IPW components (if needed)
    # --------------------------------------------------

    if mode in {"dr", "ipw"}:
        D1 = T / pi
        D0 = (1 - T) / (1 - pi)

        if mode == "dr":
            N1 = T * (Y - mu1) / pi
            N0 = (1 - T) * (Y - mu0) / (1 - pi)
        else:  # IPW only
            N1 = Y * D1
            N0 = Y * D0

        mN1 = N1.mean()
        mN0 = N0.mean()
        mD1 = D1.mean()
        mD0 = D0.mean()

    # --------------------------------------------------
    # Point estimate
    # --------------------------------------------------

    eps = 1e-12
    if mode == "om":
        ATE = base.mean()

    elif stabilized:
        term_1 = mN1 / (mD1 + eps)
        term_0 = mN0 / (mD0 + eps)
        ATE = base.mean() + term_1 - term_0

    else:
        term_1 = mN1
        term_0 = mN0
        ATE = base.mean() + term_1 - term_0

    # --------------------------------------------------
    # Variance (delta method)
    # --------------------------------------------------

    if mode == "om":
        # simple variance of base
        var_hat = base.var(ddof=1) / n

    elif stabilized:
        Z = np.column_stack([base, N1, D1, N0, D0])
        g = np.array([
            1.0,
            1.0 / (mD1 + eps),
            -mN1 / ((mD1 + eps)**2),
            -1.0 / (mD0 + eps),
            mN0 / ((mD0 + eps)**2),
        ])
        S = np.cov(Z, rowvar=False, ddof=1)
        var_hat = (g @ S @ g) / n

    else:
        Z = np.column_stack([base, N1, N0])
        g = np.array([1.0, 1.0, -1.0])
        S = np.cov(Z, rowvar=False, ddof=1)
        var_hat = (g @ S @ g) / n

    STD = float(np.sqrt(max(var_hat, 0.0)))

    return {"ATE": ATE, "STD": STD}

# 7 Generate Results

In [35]:
def process_pair(pair):

    c = pair[0]
    e = pair[1]

    ate_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=0.01,
        drop_inverted=True,
        stabilized=False,
        mode="ipw",
    )

    ate_dr_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=0.01,
        drop_inverted=True,
        stabilized=False,        
        mode="dr",
    )

    ate_om_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=0.01,
        drop_inverted=True,
        stabilized=False,        
        mode="om",
    )
    
    abl_dict = get_ATE(
        cause_item=c, 
        effect_item=e,
        clip=0.5,
        drop_inverted=True,
        stabilized=False,
        mode="ipw",
    )

    ate_stabilized_dict = get_ATE(
        cause_item=c,
        effect_item=e, 
        clip=0.01,
        drop_inverted=True,
        stabilized=True,
        mode="ipw",
    )

    return {
        "cause_id": pair[0],
        "effect_id": pair[1],
        "ATE": ate_dict["ATE"],
        "STD": ate_dict["STD"],
        "ATE_DR": ate_dr_dict["ATE"],
        "STD_DR": ate_dr_dict["STD"],
        "ATE_OM": ate_om_dict["ATE"],
        "STD_OM": ate_om_dict["STD"],
        "ABLT": abl_dict["ATE"],
        "STD_ABLT": abl_dict["STD"],
        "ATE_STABILIZED": ate_stabilized_dict["ATE"],
        "STD_STABILIZED": ate_stabilized_dict["STD"],
        "cosine_similarity": cosimilarity(*pair),
        "correlation": correlation(*pair),
        "diff_of_conditionals": diff_of_conditionals(*pair),
        "jacard_index": jacard_index(*pair),
        "sasrec_score": sasrec_scores[pair],
    }

In [36]:
all_results = []
for pair in tqdm(chosen_ids):
    results = process_pair(pair)
    all_results.append(results)

raw_results = pd.DataFrame(all_results)

  0%|          | 0/7340 [00:00<?, ?it/s]

In [37]:
merged = pd.merge(
    left=oracle,
    right=raw_results,
    left_index=True,
    right_index=True,
)

merged.to_csv(base_artifacts / 'Datasets' / 'Sequels' / 'sequels_evaluated.csv', index=False)